<a href="https://colab.research.google.com/github/Zeshan811/StarterNotebookA1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}
print("Connected.")

Paste your Hugging Face READ token (hf_...): ··········
Connected.


In [2]:
feature_frame = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_avg_position) AS avg_position_month,
        (SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)) AS ctr_month
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01' AND report_date < DATE '2026-04-01'
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

print(feature_frame.shape)
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176738, 5)


,content_hash_id,total_impressions,total_clicks,avg_position_month,ctr_month
0,content_d0dff76c889de68f,181.0,0.0,5.147402,0.000000
1,content_67741cce996cfafa,46.0,1.0,4.828125,0.021739
2,content_2e6360ad20fd7107,899.0,1.0,5.145765,0.001112
3,content_ac8663da7484669a,34.0,0.0,4.909314,0.000000
4,content_65c50dfe9d87a585,3108.0,0.0,6.969536,0.000000


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*



**The rule, in plain words:**
Flag pages that get enough traffic to matter (impressions ≥ 100), sit in a
"fixable" position range (not already #1-3, which is near-optimal; not
buried past position 20, which is a demand problem, not a CTR problem),
and whose click-through rate is below what's typical for their position
tier. These pages are visible but under-converting clicks — a metadata
or snippet review is a realistic, low-cost action.

**Reason codes this rule can output:**
- `ctr_review_candidate` — position 4-10, CTR below that tier's average
- `page_one_decay_risk` — position 11-20, CTR below that tier's average,
  and impressions high enough that the gap represents real lost clicks

**Action label:** `review_title_and_meta`

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
import numpy as np

def position_tier(p):
    if p <= 3: return "1-3"
    elif p <= 10: return "4-10"
    elif p <= 20: return "11-20"
    else: return "21+"

feature_frame["position_tier"] = feature_frame["avg_position_month"].apply(position_tier)

tier_avg_ctr = feature_frame.groupby("position_tier")["ctr_month"].mean().to_dict()
print("Tier average CTR:", tier_avg_ctr)

feature_frame["tier_avg_ctr"] = feature_frame["position_tier"].map(tier_avg_ctr)
feature_frame["ctr_gap"] = feature_frame["tier_avg_ctr"] - feature_frame["ctr_month"]

Tier average CTR: {'1-3': 0.012399437824607349, '11-20': 0.0032111899695643977, '21+': 0.0019276409866961944, '4-10': 0.0049260523876178365}


In [4]:
candidates = feature_frame[
    (feature_frame["total_impressions"] >= 100) &
    (feature_frame["avg_position_month"] > 3) &
    (feature_frame["avg_position_month"] <= 20) &
    (feature_frame["ctr_gap"] > 0)
].copy()

def assign_reason(row):
    if 4 <= row["avg_position_month"] <= 10:
        return "ctr_review_candidate"
    else:
        return "page_one_decay_risk"

candidates["reason_code"] = candidates.apply(assign_reason, axis=1)
candidates["action"] = "review_title_and_meta"

# Score: bigger CTR gap + more impressions = bigger real-world opportunity
candidates["norm_impressions"] = (candidates["total_impressions"] - candidates["total_impressions"].min()) / \
                                   (candidates["total_impressions"].max() - candidates["total_impressions"].min())
candidates["norm_ctr_gap"] = (candidates["ctr_gap"] - candidates["ctr_gap"].min()) / \
                               (candidates["ctr_gap"].max() - candidates["ctr_gap"].min())

candidates["baseline_score"] = (0.5 * candidates["norm_impressions"] + 0.5 * candidates["norm_ctr_gap"]) * 100

ranked = candidates.sort_values("baseline_score", ascending=False)

import os
os.makedirs("work/outputs", exist_ok=True)
output_cols = ["content_hash_id", "total_impressions", "total_clicks", "avg_position_month",
               "ctr_month", "tier_avg_ctr", "ctr_gap", "baseline_score", "reason_code", "action"]
ranked[output_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Written {len(ranked)} rows to work/outputs/baseline_action_score.csv")
ranked[output_cols].head(10)

Written 52839 rows to work/outputs/baseline_action_score.csv


,content_hash_id,total_impressions,total_clicks,avg_position_month,ctr_month,tier_avg_ctr,ctr_gap,baseline_score,reason_code,action
100117,content_44f34c0a90047651,212404.0,24.0,7.346909,0.000113,0.004926,0.004813,92.210328,ctr_review_candidate,review_title_and_meta
78279,content_8e1334d6356668e3,134984.0,1.0,4.545582,0.000007,0.004926,0.004919,77.471151,ctr_review_candidate,review_title_and_meta
83616,content_34a70fea29d15f24,143019.0,43.0,3.219473,0.000301,0.004926,0.004625,76.135437,page_one_decay_risk,review_title_and_meta
166512,content_fec55986a1868d62,124075.0,1.0,9.385150,0.000008,0.004926,0.004918,75.236676,ctr_review_candidate,review_title_and_meta
150874,content_b99ea6861864dea5,194337.0,361.0,4.450106,0.001858,0.004926,0.003068,70.812042,ctr_review_candidate,review_title_and_meta
62738,content_7c6373141eae744a,132593.0,83.0,5.789019,0.000626,0.004926,0.004300,70.704093,ctr_review_candidate,review_title_and_meta
106275,content_f6116743b00afc2d,107584.0,15.0,9.536301,0.000139,0.004926,0.004787,70.535410,ctr_review_candidate,review_title_and_meta
62768,content_acbcc847f8996314,170808.0,262.0,3.361195,0.001534,0.004926,0.003392,69.292724,page_one_decay_risk,review_title_and_meta
1858,content_cd3d932d4e1c8db0,89332.0,4.0,7.786219,0.000045,0.004926,0.004881,67.768676,ctr_review_candidate,review_title_and_meta
154325,content_046fc480045b88f5,83788.0,6.0,7.289152,0.000072,0.004926,0.004854,66.364104,ctr_review_candidate,review_title_and_meta


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:
top20 = ranked.head(20)
for i, row in top20.reset_index(drop=True).iterrows():
    print(f"#{i+1}  {row['content_hash_id']}")
    print(f"    action: {row['action']} | reason: {row['reason_code']} | score: {row['baseline_score']:.1f}")
    print(f"    impressions: {row['total_impressions']:.0f}, ctr: {row['ctr_month']:.3f} "
          f"(tier avg {row['tier_avg_ctr']:.3f}), position: {row['avg_position_month']:.1f}")
    print()

#1  content_44f34c0a90047651
    action: review_title_and_meta | reason: ctr_review_candidate | score: 92.2
    impressions: 212404, ctr: 0.000 (tier avg 0.005), position: 7.3

#2  content_8e1334d6356668e3
    action: review_title_and_meta | reason: ctr_review_candidate | score: 77.5
    impressions: 134984, ctr: 0.000 (tier avg 0.005), position: 4.5

#3  content_34a70fea29d15f24
    action: review_title_and_meta | reason: page_one_decay_risk | score: 76.1
    impressions: 143019, ctr: 0.000 (tier avg 0.005), position: 3.2

#4  content_fec55986a1868d62
    action: review_title_and_meta | reason: ctr_review_candidate | score: 75.2
    impressions: 124075, ctr: 0.000 (tier avg 0.005), position: 9.4

#5  content_b99ea6861864dea5
    action: review_title_and_meta | reason: ctr_review_candidate | score: 70.8
    impressions: 194337, ctr: 0.002 (tier avg 0.005), position: 4.5

#6  content_7c6373141eae744a
    action: review_title_and_meta | reason: ctr_review_candidate | score: 70.7
    impr

For each of the top 20 — action, reason code, confidence note, what would make it wrong:

1. content_44f34c0a90047651 — review_title_and_meta | ctr_review_candidate.
   Confidence: high (212K impressions, position 7.3, essentially zero clicks).
   Would be wrong if: this is a data artifact rather than a real CTR problem
   — the near-zero CTR across nearly the entire list is unusual enough to
   warrant a raw-data spot check before trusting the ranking.

2. content_8e1334d6356668e3 — review_title_and_meta | ctr_review_candidate.
   Confidence: high (135K impressions, position 4.5, zero clicks despite a
   strong position). Would be wrong if: the clicks column undercounts for
   this content item specifically (e.g. a join/dedup issue in aggregation).

3. content_34a70fea29d15f24 — review_title_and_meta | page_one_decay_risk.
   Confidence: medium-high (position 3.2 is near-top, so zero clicks here is
   the most surprising row in the list). Would be wrong if: position 3.2 is
   an average masking days at position 1 and days far lower — an average
   position can hide real day-to-day swings.

4. content_fec55986a1868d62 — review_title_and_meta | ctr_review_candidate.
   Confidence: high (124K impressions, position 9.4, zero clicks). Would be
   wrong if: the query intent is purely informational with a featured
   snippet absorbing clicks before the page link — position doesn't
   guarantee a clickable listing.

5. content_b99ea6861864dea5 — review_title_and_meta | ctr_review_candidate.
   Confidence: high (194K impressions, small non-zero CTR of 0.002). Would
   be wrong if: this page cannibalizes clicks with a sibling URL for the
   same client — consolidation, not a CTR problem (per the lane guide's
   decline-vs-consolidation check).

6. content_7c6373141eae744a — review_title_and_meta | ctr_review_candidate.
   Confidence: high (132K impressions, position 5.8). Would be wrong if:
   the title/meta are already strong and the real issue is a mismatched
   search intent, which a title rewrite wouldn't fix.

7. content_f6116743b00afc2d — review_title_and_meta | ctr_review_candidate.
   Confidence: medium (107K impressions, position 9.5 — right at the edge
   of page one). Would be wrong if: position 9.5 sits right on a SERP
   feature boundary (e.g. "People Also Ask" box pushing this listing out
   of easy view) that a title fix can't overcome.

8. content_acbcc847f8996314 — review_title_and_meta | page_one_decay_risk.
   Confidence: medium-high (170K impressions, position 3.4). Would be wrong
   if: this is a branded/navigational query where users already know the
   destination and skip clicking through search at all.

9. content_cd3d932d4e1c8db0 — review_title_and_meta | ctr_review_candidate.
   Confidence: medium (89K impressions, position 7.8). Would be wrong if:
   impressions are inflated by irrelevant long-tail queries this page
   barely relates to, making the "low CTR" misleading.

10. content_046fc480045b88f5 — review_title_and_meta | ctr_review_candidate.
    Confidence: medium (84K impressions, position 7.3). Would be wrong if:
    the page is genuinely new/thin and low CTR reflects real low relevance,
    not a fixable metadata issue.

11. content_9540d884af3e41fd — review_title_and_meta | ctr_review_candidate.
    Confidence: medium (82K impressions, position 7.8). Would be wrong if:
    seasonal demand for this topic is fading, so the click drop reflects
    less interest, not a CTR/title problem.

12. content_f43118e089ecc69a — review_title_and_meta | ctr_review_candidate.
    Confidence: medium (139K impressions, position 5.0). Would be wrong if:
    the small non-zero CTR (0.001) is close enough to the tier average
    that this page isn't meaningfully underperforming once normal
    variation is accounted for.

13. content_425715547c6a3ea8 — review_title_and_meta | ctr_review_candidate.
    Confidence: medium (72K impressions, position 6.4). Would be wrong if:
    impression volume this month was an unusual spike (e.g. a news-driven
    query), and CTR will normalize once volume returns to baseline.

14. content_36fc1ee501ec072d — review_title_and_meta | ctr_review_candidate.
    Confidence: medium (73K impressions, position 6.5). Would be wrong if:
    this content item overlaps heavily with #13 in topic, and both are
    really one underlying opportunity counted twice.

15. content_4977e90c4d93cf9f — review_title_and_meta | ctr_review_candidate.
    Confidence: medium (74K impressions, position 7.4). Would be wrong if:
    the page recently changed and the low CTR reflects a temporary
    reindexing dip rather than a lasting problem.

16. content_945d6ff91386c817 — review_title_and_meta | ctr_review_candidate.
    Confidence: lower (58K impressions — smaller volume than rows above).
    Would be wrong if: at this lower volume, the CTR estimate is noisier
    and less reliable than for higher-impression rows.

17. content_e578ac84778da489 — review_title_and_meta | ctr_review_candidate.
    Confidence: medium (118K impressions, position 4.1, small non-zero CTR).
    Would be wrong if: position 4.1 sits just below a "People Also Ask" or
    featured snippet block that structurally caps CTR regardless of title.

18. content_471d9cabce329a66 — review_title_and_meta | ctr_review_candidate.
    Confidence: medium (165K impressions, position 4.7). Would be wrong if:
    this page ranks for a query where a competitor's brand dominates
    clicks structurally, unrelated to this page's own metadata.

19. content_bf078007df823490 — review_title_and_meta | ctr_review_candidate.
    Confidence: lower (45K impressions — near the bottom of the reviewed
    volume range). Would be wrong if: at this volume, a handful of missing
    click events could swing CTR disproportionately.

20. content_37a6fac676c8cebb — review_title_and_meta | ctr_review_candidate.
    Confidence: lower (48K impressions, position 4.4). Would be wrong if:
    this is the noisiest edge of the ranked queue — scores this close
    together (58.9 vs neighbors) could reorder with small data changes.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [7]:
# Look just below the top-20 cutoff for genuinely weak/borderline picks
borderline = ranked.iloc[15:25]
borderline[["content_hash_id", "total_impressions", "ctr_month", "avg_position_month", "baseline_score"]]

,content_hash_id,total_impressions,ctr_month,avg_position_month,baseline_score
83640,content_945d6ff91386c817,58278.0,0.000086,6.149151,61.010389
57683,content_e578ac84778da489,117764.0,0.001384,4.124146,59.980101
3201,content_471d9cabce329a66,164885.0,0.002402,4.656030,59.274644
12903,content_bf078007df823490,44707.0,0.000000,7.906249,59.109753
105871,content_37a6fac676c8cebb,48049.0,0.000083,4.406859,58.947253
83629,content_1642f339bd6e7c8d,65330.0,0.000520,3.693395,58.038768
83563,content_0c5606abaaab3178,38865.0,0.000000,5.694764,57.916685
37469,content_39e19a3ec2d95f9d,42185.0,0.000095,9.099972,57.632230
50851,content_1bb7d17cac7f6b78,55406.0,0.000361,5.359009,57.630691
132330,content_f57f0a707cf46a0a,46781.0,0.000192,6.803585,57.580501


**Weak picks:**

The single biggest concern across the entire top-20 is not any one row —
it's the pattern itself: every row shows CTR at or near 0.000, even for
pages with 100K+ impressions and strong positions (as low as 3.2). This
is far below what's normal for real search behavior, and it's suspicious
precisely because it's so consistent. Before trusting this queue, I would
sanity-check gsc_clicks directly against a known page, and re-verify the
aggregation (SUM/GROUP BY) isn't silently dropping or miscounting click
rows for this data slice. If the pattern holds after that check, it would
mean March 2026 GSC click data has unusually poor coverage — a data-limit
finding, not a genuine SEO opportunity finding.

Ranks 16-20 are also weaker on their own terms: their scores cluster
tightly (58.9-61.2) and their impression volumes are the lowest in the
top 20, meaning small data changes could reorder them.

**Leakage check — confirmed clean:**
- No FlyRank product flags used: no health_score, priority_score,
  action_type, or existing refresh/review flags anywhere in the feature
  set or the rule.
- No future-window data: every column is computed only from March 2026
  — the same month the rule is scoring, no peeking into April or beyond.
- The score is built entirely from observed GSC signals confirmed in the
  data contract (ML-04), not from any derived/label-like column.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.